# CARF Causal Analysis

This notebook demonstrates CARF's causal inference capabilities:

1. Run causal analysis with custom data
2. Interpret effect estimates and confidence intervals
3. Validate results with refutation tests
4. Check proposed actions against Guardian policies

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.api.library import run_causal, check_guardian, export_reproducibility_artifact

## 1. Load Data

Load the CSRD materiality dataset:

In [ ]:
df = pd.read_csv('../demo/data/csrd_materiality.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Run Causal Analysis

Estimate the causal effect of climate transition risk on operating costs:

In [ ]:
result = await run_causal(
    query="What is the causal effect of climate transition risk on operating cost change?",
    data=df,
    treatment="climate_transition_risk",
    outcome="operating_cost_change",
    covariates=["energy_intensity", "company_size"],
)

print(f"Effect estimate: {result.get('effect_estimate', 'N/A')}")
print(f"Confidence interval: {result.get('confidence_interval', 'N/A')}")
print(f"P-value: {result.get('p_value', 'N/A')}")
print(f"Refutation passed: {result.get('refutation_passed', 'N/A')}")
print(f"\nInterpretation: {result.get('interpretation', 'N/A')}")

## 3. Guardian Policy Check

Validate proposed actions against CARF Guardian policies:

In [ ]:
guardian_result = await check_guardian(
    proposed_action={
        "action_type": "causal_recommendation",
        "parameters": {
            "effect_size": result.get('effect_estimate', 0),
            "amount": 25000,
        }
    },
    context={"user_role": "senior", "risk_level": "MEDIUM"},
)

print(f"Guardian verdict: {guardian_result['verdict']}")
print(f"Violations: {guardian_result['violations']}")

## 4. Export for Reproducibility

In [ ]:
artifact = export_reproducibility_artifact(result)
import json
print(json.dumps(artifact, indent=2, default=str)[:1000])